
# Bokeh for Time Series Analysis
<hr style="border: 2px solid black;">


<img src="./images/bokeh.png" alt="bokeh Logo" width="1000"/>
<hr style="border: 2px solid black;">

<img src="./images/bokeh_at_ag_glance.png" alt="bokeh Logo" width="1000"/>
<hr style="border: 2px solid black;">
**Introduction to Bokeh**
Bokeh is an interactive visualization library for Python that targets modern web browsers for presentation.
Unlike Matplotlib, which is primarily designed for static plots, Bokeh excels at creating
interactive plots and dashboards. It can handle large datasets and streaming data,
making it suitable for real-time applications.

**Key Features of Bokeh:**

* **Interactivity:** Built-in support for zooming, panning, hovering, and other interactive tools.
* **Web-Focused:** Generates HTML and JavaScript, making it easy to embed plots in web pages.
* **High Performance:** Can handle large datasets efficiently.
* **Versatility:** Supports a wide range of plot types (lines, bars, scatter plots, etc.).

<hr style="border: 2px solid black;">


**Documentation:**

For comprehensive documentation, please refer to the official Bokeh website: [https://docs.bokeh.org/en/latest/](https://docs.bokeh.org/en/latest/)


<hr style="border: 2px solid black;">


**Lab Exercise:**

Your task is to recreate the time series analysis lab we previously conducted using Pandas,
Matplotlib, and Seaborn, but this time, utilize the Bokeh library for visualization.
This will involve:

1.  Loading and preprocessing the "AirPassengersDates.csv" dataset.
2.  Creating interactive Bokeh plots for:
    * Time series line plots
    * Bar plots of aggregated data
    * Visualizing mean and standard deviation
    * Outlier detection
    * Resampling (upsampling and downsampling)
    * Lag analysis
    * Autocorrelation

Pay close attention to Bokeh's features for interactivity (tools, hover effects) and
its handling of data sources. Aim to replicate the insights and visualizations
from the previous lab while leveraging Bokeh's strengths.

Good luck!
<hr style="border: 2px solid black;">

In [2]:
# --- IMPORTATIONS ---
import pandas as pd
import numpy as np
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, Span
from statsmodels.tsa.stattools import acf
import warnings
warnings.filterwarnings('ignore')

# Commande magique pour afficher Bokeh dans le notebook VS Code
output_notebook()

# --- SECTION 1 & 2 : CHARGEMENT ET DATES (Avec l'URL de secours infaillible) ---
print("1. Chargement et préparation des données...")
url = "https://raw.githubusercontent.com/AileenNielsen/TimeSeriesAnalysisWithPython/master/data/AirPassengers.csv"
df = pd.read_csv(url)
df.columns = ["Date", "#Passengers"] # On force les bons noms
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month


# --- SECTION 3 : VISUALISATIONS AVEC BOKEH ---
print("Génération des graphiques interactifs avec Bokeh...")

# 1. Plot de base (Ligne du temps)
p1 = figure(title="1. Air Passengers Over Time (survoler avec la souris !)", x_axis_type="datetime", width=800, height=350)
p1.line(df['Date'], df['#Passengers'], line_width=2, color="navy")
# Ajout de l'interactivité au survol
p1.add_tools(HoverTool(tooltips=[("Date", "@x{%F}"), ("Passagers", "@y")], formatters={'@x': 'datetime'}))
show(p1)

# 2. Aggregation (Bar Plot)
monthly = df.groupby("Month")["#Passengers"].sum().reset_index()
p2 = figure(title="2. Total Passengers per Month", width=800, height=350)
p2.vbar(x=monthly['Month'], top=monthly['#Passengers'], width=0.8, color="teal", alpha=0.8)
show(p2)

# 3. Moyenne et Écart-Type (Avec Span)
mean_val = df["#Passengers"].mean()
std_val = df["#Passengers"].std()

p3 = figure(title="3. Passengers with Mean and Standard Deviation", x_axis_type="datetime", width=800, height=350)
p3.line(df['Date'], df['#Passengers'], legend_label="Passengers", line_width=2)
# Ajout des lignes horizontales
p3.add_layout(Span(location=mean_val, dimension='width', line_color='red', line_dash='dashed', line_width=2))
p3.add_layout(Span(location=mean_val + std_val, dimension='width', line_color='green', line_dash='dashed'))
p3.add_layout(Span(location=mean_val - std_val, dimension='width', line_color='green', line_dash='dashed'))
show(p3)


# --- SECTION 4 : OUTLIERS (Valeurs aberrantes) ---
df["Z-Score"] = (df["#Passengers"] - mean_val) / std_val
df["Abs_Z"] = abs(df["Z-Score"])
outliers = df[df["Abs_Z"] > 2]

p4 = figure(title="4. Outlier Detection (Z-Score > 2)", x_axis_type="datetime", width=800, height=350)
p4.line(df['Date'], df['#Passengers'], legend_label="Passengers", color="blue")
p4.scatter(outliers['Date'], outliers['#Passengers'], color="red", size=8, legend_label="Outliers")
show(p4)


# --- SECTION 5 : RESAMPLING (Downsampling Annuel) ---
# On passe en index pour le resampling Pandas
df.set_index("Date", inplace=True)
yearly = df.resample("YE")["#Passengers"].mean() # 'YE' pour Year End
df.reset_index(inplace=True)
yearly = yearly.reset_index()

p5 = figure(title="5. Downsampling to Yearly Average", x_axis_type="datetime", width=800, height=350)
p5.line(df['Date'], df['#Passengers'], alpha=0.4, legend_label="Original Monthly", color="gray")
p5.line(yearly['Date'], yearly['#Passengers'], color="orange", line_width=3, legend_label="Yearly Average")
p5.scatter(yearly['Date'], yearly['#Passengers'], color="orange", size=8)
show(p5)


# --- SECTION 6 : LAG ANALYSIS (Shift) ---
df["Shifted"] = df["#Passengers"].shift(1)

p6 = figure(title="6. Lag Analysis (Shift de 1 période)", x_axis_type="datetime", width=800, height=350)
p6.line(df['Date'], df['#Passengers'], legend_label="Original", color="blue")
p6.line(df['Date'], df["Shifted"], legend_label="Shifted (-1)", color="red", line_dash="dotted", line_width=2)
show(p6)


# --- SECTION 7 : AUTOCORRÉLATION (ACF) ---
# Calcul de l'ACF avec statsmodels
acf_vals = acf(df['#Passengers'].dropna(), nlags=30)
lags = np.arange(len(acf_vals))

p7 = figure(title="7. Autocorrelation Function (ACF)", width=800, height=350)
# Dans Bokeh, on utilise un mix de vbar et scatter pour imiter le plot_acf
p7.vbar(x=lags, top=acf_vals, width=0.1, color="black")
p7.scatter(lags, acf_vals, size=8, color="navy")
show(p7)


Loading BokehJS ...

1. Chargement et préparation des données...
Génération des graphiques interactifs avec Bokeh...
